<a href="https://colab.research.google.com/github/aditya-kumar-seth/cv/blob/main/conv_filter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# convolution scratch

In [ ]:
import numpy as np

image = np.array([
    [1, 2, 3, 0, 1],
    [4, 5, 6, 1, 2],
    [7, 8, 9, 2, 3],
    [1, 2, 3, 4, 5],
    [6, 7, 8, 9, 0]
])

kernel = np.array([
    [1, 0, -1],
    [1, 0, -1],
    [1, 0, -1]
])

stride = 1
padding = 1

# Add padding
padded_image = np.pad(image, pad_width=padding, mode='constant', constant_values=0)

image_height, image_width = padded_image.shape
kernel_height, kernel_width = kernel.shape

output_height = ((image_height - kernel_height) // stride) + 1
output_width = ((image_width - kernel_width) // stride) + 1

output = np.zeros((output_height, output_width))

for i in range(0, output_height):
    for j in range(0, output_width):

        region = padded_image[
            i*stride : i*stride + kernel_height,
            j*stride : j*stride + kernel_width
        ]

        output[i][j] = np.sum(region * kernel)

print("Padded Image:")
print(padded_image)

print("\nOutput Feature Map:")
print(output)

# Filters and noise

In [ ]:
import cv2
import numpy as np

# Read image
image = cv2.imread("image.jpg")

# Check if image loaded properly
if image is None:
    print("Error: Image not found")
    exit()

# Resize image
image = cv2.resize(image, (512, 512))

# Convert to float32 and normalize
image_float = image.astype(np.float32) / 255.0

# Convert to grayscale
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# Normalize grayscale image
gray_float = gray.astype(np.float32) / 255.0

# -----------------------------
# Average Blur
# -----------------------------
average_blur = cv2.blur(image_float, (5, 5))

# -----------------------------
# Gaussian Blur
# -----------------------------
gaussian_blur = cv2.GaussianBlur(image_float, (5, 5), 0)

# -----------------------------
# Median Blur
# Median blur needs uint8 image
# -----------------------------
median_blur = cv2.medianBlur((image_float * 255).astype(np.uint8), 5)

# -----------------------------
# Bilateral Filter
# Bilateral filter works better on uint8
# -----------------------------
bilateral_filter = cv2.bilateralFilter(
    (image_float * 255).astype(np.uint8),
    9,
    75,
    75
)

# -----------------------------
# Add Gaussian Noise
# -----------------------------
mean = 0
std_dev = 0.05

noise = np.random.normal(mean, std_dev, gray_float.shape).astype(np.float32)

noisy_image = gray_float + noise

# Clip values to valid range [0,1]
noisy_image = np.clip(noisy_image, 0.0, 1.0)

# Convert noisy image to uint8 for filtering
noisy_uint8 = (noisy_image * 255).astype(np.uint8)

# -----------------------------
# Noise Removal
# -----------------------------
gaussian_denoised = cv2.GaussianBlur(noisy_uint8, (5, 5), 0)
median_denoised = cv2.medianBlur(noisy_uint8, 5)
bilateral_denoised = cv2.bilateralFilter(noisy_uint8, 9, 75, 75)

# -----------------------------
# Sharpening Filter
# -----------------------------
sharpen_kernel = np.array([
    [0, -1, 0],
    [-1, 5, -1],
    [0, -1, 0]
], dtype=np.float32)

sharpened = cv2.filter2D(image_float, -1, sharpen_kernel)

# -----------------------------
# Edge Detection
# -----------------------------
sobel_x = cv2.Sobel(gray_float, cv2.CV_32F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(gray_float, cv2.CV_32F, 0, 1, ksize=3)
laplacian = cv2.Laplacian(gray_float, cv2.CV_32F)

# Convert grayscale image to uint8 for Canny
gray_uint8 = (gray_float * 255).astype(np.uint8)

canny_edges = cv2.Canny(gray_uint8, 100, 200)

# -----------------------------
# Display Images
# -----------------------------
cv2.imshow("Original Image", image)
cv2.imshow("Average Blur", average_blur)
cv2.imshow("Gaussian Blur", gaussian_blur)
cv2.imshow("Median Blur", median_blur)
cv2.imshow("Bilateral Filter", bilateral_filter)

cv2.imshow("Noisy Image", noisy_uint8)
cv2.imshow("Gaussian Denoised", gaussian_denoised)
cv2.imshow("Median Denoised", median_denoised)
cv2.imshow("Bilateral Denoised", bilateral_denoised)

cv2.imshow("Sharpened Image", sharpened)

cv2.imshow("Sobel X", sobel_x)
cv2.imshow("Sobel Y", sobel_y)
cv2.imshow("Laplacian", laplacian)
cv2.imshow("Canny Edges", canny_edges)

cv2.waitKey(0)
cv2.destroyAllWindows()

# add noise and filter it

In [ ]:
import cv2
import numpy as np

# Read image in grayscale
image = cv2.imread("image.jpg", cv2.IMREAD_GRAYSCALE)

# Resize image
image = cv2.resize(image, (512, 512))

# Normalize image
image = image.astype(np.float32) / 255.0

# -------------------------------------------------
# Add Gaussian Noise
# -------------------------------------------------
mean = 0
std_dev = 0.08

gaussian_noise = np.random.normal(mean, std_dev, image.shape).astype(np.float32)

gaussian_noisy_image = image + gaussian_noise
gaussian_noisy_image = np.clip(gaussian_noisy_image, 0.0, 1.0)

# -------------------------------------------------
# Add Salt and Pepper Noise
# -------------------------------------------------
salt_pepper_image = image.copy()

prob = 0.02

random_matrix = np.random.rand(*image.shape)

salt_pepper_image[random_matrix < prob] = 0.0
salt_pepper_image[random_matrix > 1 - prob] = 1.0

# -------------------------------------------------
# Add Speckle Noise
# -------------------------------------------------
speckle_noise = np.random.randn(*image.shape).astype(np.float32)

speckle_noisy_image = image + image * speckle_noise * 0.2
speckle_noisy_image = np.clip(speckle_noisy_image, 0.0, 1.0)

# -------------------------------------------------
# Convert noisy images to uint8
# -------------------------------------------------
gaussian_noisy_uint8 = (gaussian_noisy_image * 255).astype(np.uint8)
salt_pepper_uint8 = (salt_pepper_image * 255).astype(np.uint8)
speckle_noisy_uint8 = (speckle_noisy_image * 255).astype(np.uint8)

# -------------------------------------------------
# Noise Removal
# -------------------------------------------------

# For Gaussian Noise
gaussian_removed = cv2.GaussianBlur(gaussian_noisy_uint8, (5, 5), 0)

# For Salt and Pepper Noise
salt_pepper_removed = cv2.medianBlur(salt_pepper_uint8, 5)

# For Speckle Noise
speckle_removed = cv2.bilateralFilter(speckle_noisy_uint8, 9, 75, 75)

# -------------------------------------------------
# Display Images
# -------------------------------------------------
cv2.imshow("Original Image", (image * 255).astype(np.uint8))

cv2.imshow("Gaussian Noise", gaussian_noisy_uint8)
cv2.imshow("Gaussian Noise Removed", gaussian_removed)

cv2.imshow("Salt and Pepper Noise", salt_pepper_uint8)
cv2.imshow("Salt and Pepper Removed", salt_pepper_removed)

cv2.imshow("Speckle Noise", speckle_noisy_uint8)
cv2.imshow("Speckle Noise Removed", speckle_removed)

cv2.waitKey(0)
cv2.destroyAllWindows()

# Alpha trimmed, contrharmonic, harmonic

In [ ]:
import cv2
import numpy as np

# Read image in grayscale
image = cv2.imread("image.jpg", cv2.IMREAD_GRAYSCALE)

# Resize image
image = cv2.resize(image, (256, 256))

# Convert to float32
image = image.astype(np.float32)

# ---------------------------------------------------
# Harmonic Mean Filter
# Formula: (m*n) / sum(1/pixel)
# ---------------------------------------------------
def harmonic_filter(img, kernel_size=3):
    pad = kernel_size // 2
    padded = np.pad(img, pad, mode='reflect')

    output = np.zeros_like(img, dtype=np.float32)

    for i in range(img.shape[0]):
        for j in range(img.shape[1]):
            region = padded[i:i+kernel_size, j:j+kernel_size]

            region = np.where(region == 0, 0.0001, region)

            output[i, j] = (kernel_size * kernel_size) / np.sum(1.0 / region)

    return np.clip(output, 0, 255).astype(np.uint8)

# ---------------------------------------------------
# Contra Harmonic Mean Filter
# Formula: sum(pixel^(Q+1)) / sum(pixel^Q)
# Q > 0 removes pepper noise
# Q < 0 removes salt noise
# ---------------------------------------------------
def contra_harmonic_filter(img, kernel_size=3, Q=1.5):
    pad = kernel_size // 2
    padded = np.pad(img, pad, mode='reflect')

    output = np.zeros_like(img, dtype=np.float32)

    for i in range(img.shape[0]):
        for j in range(img.shape[1]):
            region = padded[i:i+kernel_size, j:j+kernel_size]

            numerator = np.sum(np.power(region, Q + 1))
            denominator = np.sum(np.power(region, Q))

            if denominator != 0:
                output[i, j] = numerator / denominator
            else:
                output[i, j] = 0

    return np.clip(output, 0, 255).astype(np.uint8)

# ---------------------------------------------------
# Alpha Trimmed Mean Filter
# Removes highest and lowest d/2 values
# ---------------------------------------------------
def alpha_trimmed_filter(img, kernel_size=3, d=2):
    pad = kernel_size // 2
    padded = np.pad(img, pad, mode='reflect')

    output = np.zeros_like(img, dtype=np.float32)

    for i in range(img.shape[0]):
        for j in range(img.shape[1]):
            region = padded[i:i+kernel_size, j:j+kernel_size].flatten()

            sorted_region = np.sort(region)

            trimmed = sorted_region[d//2 : len(sorted_region) - d//2]

            output[i, j] = np.mean(trimmed)

    return np.clip(output, 0, 255).astype(np.uint8)

# ---------------------------------------------------
# Geometric Mean Filter
# Formula: product(pixel)^(1/(m*n))
# ---------------------------------------------------
def geometric_filter(img, kernel_size=3):
    pad = kernel_size // 2
    padded = np.pad(img, pad, mode='reflect')

    output = np.zeros_like(img, dtype=np.float32)

    for i in range(img.shape[0]):
        for j in range(img.shape[1]):
            region = padded[i:i+kernel_size, j:j+kernel_size]

            region = np.where(region == 0, 0.0001, region)

            product = np.prod(region)
            output[i, j] = product ** (1.0 / (kernel_size * kernel_size))

    return np.clip(output, 0, 255).astype(np.uint8)

# ---------------------------------------------------
# Apply Filters
# ---------------------------------------------------
harmonic_result = harmonic_filter(image, 3)

contra_result = contra_harmonic_filter(image, 3, Q=1.5)

alpha_trimmed_result = alpha_trimmed_filter(image, 3, d=2)

geometric_result = geometric_filter(image, 3)

# ---------------------------------------------------
# Display Images
# ---------------------------------------------------
cv2.imshow("Original Image", image.astype(np.uint8))
cv2.imshow("Harmonic Filter", harmonic_result)
cv2.imshow("Contra Harmonic Filter", contra_result)
cv2.imshow("Alpha Trimmed Filter", alpha_trimmed_result)
cv2.imshow("Geometric Mean Filter", geometric_result)

cv2.waitKey(0)
cv2.destroyAllWindows()

# Canny

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Read image
image = cv2.imread("image.jpg")

# Convert BGR to RGB for visualization
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Convert to grayscale
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# Normalize grayscale image
gray = gray.astype(np.float32) / 255.0

# Convert back to uint8 because Canny expects uint8
gray_uint8 = (gray * 255).astype(np.uint8)

# Apply Gaussian Blur before Canny
blurred = cv2.GaussianBlur(gray_uint8, (5, 5), 1.4)

# Apply Canny Edge Detection
canny_edges = cv2.Canny(blurred, threshold1=100, threshold2=200)

# Visualization
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.imshow(image_rgb)
plt.title("Original Image")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(blurred, cmap="gray")
plt.title("Blurred Grayscale")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(canny_edges, cmap="gray")
plt.title("Canny Edges")
plt.axis("off")

plt.show()

# autoencoder and decoder

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.datasets import mnist
import matplotlib.pyplot as plt
import numpy as np

# Load MNIST dataset
(x_train, _), (x_test, _) = mnist.load_data()

# Normalize images
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Flatten 28x28 images into 784-dimensional vectors
x_train = x_train.reshape((len(x_train), 784))
x_test = x_test.reshape((len(x_test), 784))

# --------------------------------------------------
# Encoder
# --------------------------------------------------
input_img = Input(shape=(784,))

encoded = Dense(256, activation='relu')(input_img)
encoded = Dense(128, activation='relu')(encoded)
encoded = Dense(64, activation='relu')(encoded)

# --------------------------------------------------
# Decoder
# --------------------------------------------------
decoded = Dense(128, activation='relu')(encoded)
decoded = Dense(256, activation='relu')(decoded)
decoded = Dense(784, activation='sigmoid')(decoded)

# Autoencoder Model
autoencoder = Model(input_img, decoded)

# Separate Encoder Model
encoder = Model(input_img, encoded)

# --------------------------------------------------
# Decoder Model
# --------------------------------------------------
encoded_input = Input(shape=(64,))

decoder_layer1 = autoencoder.layers[-3](encoded_input)
decoder_layer2 = autoencoder.layers[-2](decoder_layer1)
decoder_output = autoencoder.layers[-1](decoder_layer2)

decoder = Model(encoded_input, decoder_output)

# Compile Model
autoencoder.compile(optimizer='adam', loss='binary_crossentropy')

# Train Model
autoencoder.fit(
    x_train,
    x_train,
    epochs=10,
    batch_size=256,
    shuffle=True,
    validation_data=(x_test, x_test)
)

# Encode Test Images
encoded_images = encoder.predict(x_test)

# Decode Images
decoded_images = decoder.predict(encoded_images)

# Reshape for visualization
original_images = x_test.reshape(-1, 28, 28)
reconstructed_images = decoded_images.reshape(-1, 28, 28)

# --------------------------------------------------
# Visualization
# --------------------------------------------------
n = 10

plt.figure(figsize=(20, 4))

for i in range(n):
    # Original Image
    plt.subplot(2, n, i + 1)
    plt.imshow(original_images[i], cmap='gray')
    plt.title("Original")
    plt.axis('off')

    # Reconstructed Image
    plt.subplot(2, n, i + 1 + n)
    plt.imshow(reconstructed_images[i], cmap='gray')
    plt.title("Reconstructed")
    plt.axis('off')

plt.show()

#